In [1]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
import sys
import os
from datetime import datetime

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
project_root_str = str(project_root)

if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

os.chdir(project_root_str)

from src.utils.team_info import *
from src.utils.helper_functions import *
from src.pipeline.props_pipeline.ppm_pipeline import *
from src.pipeline.props_pipeline.apm_pipeline import *
from src.pipeline.props_pipeline.rpm_pipeline import *
from src.pipeline.props_pipeline.min_pipeline import *
from src.live import *
from src.historical_analysis.dataScraper import *

warnings.filterwarnings("ignore")
pd.set_option('display.max_columns', None)

### Get updated lineups

In [2]:
from src.utils.scrap_starters import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
print("\nQuestionable Players:")
print(scraper.getQuestionablePlayers())
print("\nOut Players:")
print(scraper.getOutPlayers())
outPlayers = scraper.getOutPlayers()
scraper.updateTeamInfo()  # Update teamInfo.py


Questionable Players:
No data available. Run getDict() first.
{}

Out Players:
No data available. Run getDict() first.
{}
No data available. Run getDict() first.
No data available. Run getDict() first.


### Dataset

In [4]:
s25 = pd.read_csv('data/raw/season_stats/S25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('data/raw/season_stats/S26.csv').sort_values(by='GAME_DATE')
p26 = pd.read_csv('data/raw/playoff_stats/P26.csv').sort_values(by='GAME_DATE')
base_df = pd.concat([s26, p26])
base_df.tail()

,Unnamed: 0,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT,E_OFF_RATING,OFF_RATING,sp_work_OFF_RATING,E_DEF_RATING,DEF_RATING,sp_work_DEF_RATING,E_NET_RATING,NET_RATING,sp_work_NET_RATING,AST_PCT,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,REB_PCT,TM_TOV_PCT,E_TOV_PCT,EFG_PCT,TS_PCT,USG_PCT,E_USG_PCT,E_PACE,PACE,PACE_PER40,sp_work_PACE,PIE,POSS,FGM_PG,FGA_PG,START_POSITION,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_TOV,TEAM_STL,TEAM_BLK,TEAM_BLKA,TEAM_PF,TEAM_PFD,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_E_OFF_RATING,TEAM_OFF_RATING,TEAM_E_DEF_RATING,TEAM_DEF_RATING,TEAM_E_NET_RATING,TEAM_NET_RATING,TEAM_AST_PCT,TEAM_AST_TO,TEAM_AST_RATIO,TEAM_OREB_PCT,TEAM_DREB_PCT,TEAM_REB_PCT,TEAM_TM_TOV_PCT,TEAM_EFG_PCT,TEAM_TS_PCT,TEAM_E_PACE,TEAM_PACE,TEAM_PACE_PER40,TEAM_POSS,TEAM_PIE,OPP_TEAM_ID,OPP_OPP_ABBREVIATION_base,OPP_OPP_NAME_base,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_FG3M,OPP_FG3A,OPP_FG3_PCT,OPP_FTM,OPP_FTA,OPP_FT_PCT,OPP_OREB,OPP_DREB,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,OPP_PLUS_MINUS,OPP_E_OFF_RATING,OPP_OFF_RATING,OPP_E_DEF_RATING,OPP_DEF_RATING,OPP_E_NET_RATING,OPP_NET_RATING,OPP_AST_PCT,OPP_AST_TO,OPP_AST_RATIO,OPP_OREB_PCT,OPP_DREB_PCT,OPP_REB_PCT,OPP_TM_TOV_PCT,OPP_EFG_PCT,OPP_TS_PCT,OPP_E_PACE,OPP_PACE,OPP_PACE_PER40,OPP_POSS,OPP_PIE,IS_PLAYOFF,POS,AGE,name
57,57,2025-26,1630540,Miles McBride,Miles,1610612752,NYK,New York Knicks,42500125,2026-04-28T00:00:00,NYK vs. ATL,W,20.033333,0,4,0.000,0,2,0.000,0,0,0.000,1,2,3,1,1,0,0,1,2,0,0,19,4.1,0,0,4.0,1,20:02,1,143.5,142.9,142.9,101.5,100.0,100.0,42.1,42.9,42.9,0.05,1.0,16.7,0.063,0.087,0.077,16.7,16.7,0.000,0.000,0.104,0.109,98.48,99.43,82.86,99.43,-0.043,42,0.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45,79,0.570,11,26,0.423,25,34,0.735,8,40,48,26,11.0,6,6,2,23,26,126,29.0,130.0,134.0,101.6,103.2,28.4,30.9,0.578,2.36,19.5,0.325,0.824,0.604,0.117,0.639,0.670,96.2,94.0,78.33,94,0.661,1610612737,ATL,Atlanta Hawks,37,83,0.446,13,42,0.310,10,17,0.588,5,22,27,25,10.0,7,2,6,26,23,97,-29.0,101.6,103.2,130.0,134.0,-28.4,-30.9,0.676,2.50,19.5,0.176,0.675,0.396,0.106,0.524,0.536,96.2,94.0,78.33,94,0.339,1,SG,25.0,Miles McBride
58,58,2025-26,1631248,Baylor Scheierman,Baylor,1610612738,BOS,Boston Celtics,42500115,2026-04-28T00:00:00,BOS vs. PHI,L,8.650000,1,2,0.500,1,2,0.500,0,0,0.000,0,0,0,0,1,0,0,0,1,0,3,-3,2.0,0,0,4.0,1,8:39,1,101.6,83.3,83.3,114.2,105.9,105.9,-12.6,-22.5,-22.5,0.00,0.0,0.0,0.000,0.000,0.000,33.3,33.3,0.750,0.750,0.136,0.160,84.68,97.11,80.92,97.11,0.000,18,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,36,89,0.404,11,39,0.282,14,23,0.609,15,34,49,25,12.0,6,0,3,17,20,97,-16.0,100.9,103.2,121.3,120.2,-20.4,-17.0,0.694,2.08,18.4,0.300,0.829,0.515,0.128,0.466,0.489,94.6,94.0,78.33,94,0.404,1610612755,PHI,Philadelphia 76ers,39,78,0.500,15,42,0.357,20,23,0.870,5,42,47,24,10.0,5,3,0,20,17,113,16.0,121.3,120.2,100.9,103.2,20.4,17.0,0.615,2.40,19.2,0.171,0.700,0.485,0.106,0.596,0.641,94.6,94.0,78.33,94,0.596,1,SG,25.0,Baylor Scheierman
59,59,2025-26,1642278,Tyler Kolek,Tyler,1610612752,NYK,New York Knicks,42500125,2026-04-28T00:00:00,NYK vs. ATL,W,3.466667,0,1,0.000,0,0,0.000,0,0,0.000,0,1,1,2,0,0,0,0,0,0,0,2,4.2,0,0,3.0,1,3:28,1,147.9,125.0,125.0,100.0,100.0,100.0,47.9,25.0,25.0,0.50,0.0,66.7,0.000,0.250,0.143,0.0,0.0,0.000,0.000,0.111,0.129,102.18,110.77,92.31,110.77,0.100,8,0.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45,79,0.570,11,26,0.423,25,34,0.735,8,40,48,26,11.0,6,6,2,23,26,126,29.0,130.0,134.0,101.6,103.2,28.4,30

### Load latest odds on file

In [5]:
def get_latest_file(pattern):
    files = list(Path('data/raw/team_lines').glob(pattern))
    return max(files, key=lambda f: f.stat().st_mtime) if files else None

file = get_latest_file('NBA_*.json')
if file is None:
    raise ValueError("No JSON file found")

# try normal load first
try:
    team_dds = pd.read_json(file)
except ValueError:
    # fallback for nested JSON
    import json
    with open(file) as f:
        data = json.load(f)
    team_dds = pd.json_normalize(data)

print("Loaded:", file.name)
team_dds.head()

Loaded: NBA_20260428_230331.json


,home_team,away_team,commence_time,bookmakers
0,Detroit Pistons,Orlando Magic,2026-04-29 23:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
1,Cleveland Cavaliers,Toronto Raptors,2026-04-29 23:40:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
2,Los Angeles Lakers,Houston Rockets,2026-04-30 02:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."
3,Atlanta Hawks,New York Knicks,2026-04-30 23:10:00+00:00,"[{'bookmaker': 'FanDuel', 'last_updated': '202..."
4,Philadelphia 76ers,Boston Celtics,2026-05-01 00:10:00+00:00,"[{'bookmaker': 'DraftKings', 'last_updated': '..."


In [6]:
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

def get_latest_file(pattern):
    files = list(Path('data/raw/player_lines').glob(pattern))
    if not files:
        return None
    return max(files, key=lambda p: p.stat().st_mtime)

us_file = get_latest_file(f'NBA_US_{today}*.csv')
dfs_file = get_latest_file(f'NBA_DFS_{today}*.csv')

#load season stats
pts_df = pd.read_csv('data/processed/training/S26_TRAINING_PPM.csv')
ast_df = pd.read_csv('data/processed/training/S26_TRAINING_APM.csv')
reb_df = pd.read_csv('data/processed/training/S26_TRAINING_RPM.csv')
min_df = pd.read_csv('data/processed/training/S26_TRAINING_MIN.csv')
pts_ast_df = pd.read_csv('data/processed/training/S26_TRAINING_PAPM.csv')

#load dfs lines
lines_dfs = pd.read_csv(dfs_file)
lines_dfs_pts = lines_dfs[(lines_dfs['CATEGORY'] == 'player_points')]
lines_dfs_ast = lines_dfs[(lines_dfs['CATEGORY'] == 'player_assists')]
lines_dfs_reb = lines_dfs[(lines_dfs['CATEGORY'] == 'player_rebounds')]
pts_names = lines_dfs_pts['NAME'].unique()
ast_names = lines_dfs_ast['NAME'].unique()
reb_names = lines_dfs_reb['NAME'].unique()

#load us lines with actual odds
lines_us = pd.read_csv(us_file)
lines_us_pts = lines_us[(lines_us['CATEGORY'] == 'player_points')]
lines_us_ast = lines_us[(lines_us['CATEGORY'] == 'player_assists')]
lines_us_reb = lines_us[(lines_us['CATEGORY'] == 'player_rebounds')]
print(f"DFS latest pull: {lines_dfs['DATA_PULLED_AT'].max()}")
print(f"US latest pull: {lines_us['DATA_PULLED_AT'].max()}")

lines_dfs_pts.head()

DFS latest pull: 2026-04-28 23:03:31
US latest pull: 2026-04-28 23:03:03


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE,DATA_PULLED_AT
0,PrizePicks,player_points,Cade Cunningham,Over,28.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31
1,PrizePicks,player_points,Cade Cunningham,Under,28.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31
2,PrizePicks,player_points,Paolo Banchero,Over,21.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31
3,PrizePicks,player_points,Paolo Banchero,Under,21.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31
4,PrizePicks,player_points,Desmond Bane,Over,18.5,-137,2026-04-29,2026-04-29T06:03:10Z,2026-04-28 23:03:31


### Load my models

In [7]:
import joblib

#minutes
min_bundle = joblib.load("src/models/saved_models/min_quantile_xgb_2026-01-02.joblib")
min_quantile_models = min_bundle["quantile_models"]
min_feature_names = min_bundle["feature_names"]

#points per minute
ppm_bundle = joblib.load("src/models/saved_models/ppm_quantile_xgb_2026-01-01.joblib")
ppm_quantile_models = ppm_bundle["quantile_models"]
ppm_feature_names = ppm_bundle["feature_names"]

#assists per minute
apm_bundle = joblib.load("src/models/saved_models/apm_quantile_xgb_2026-01-01.joblib")
apm_quantile_models = apm_bundle["quantile_models"]
apm_feature_names = apm_bundle["feature_names"]

#rebounds per minute
rpm_bundle = joblib.load("src/models/saved_models/rpm_quantile_xgb_2026-01-01.joblib")
rpm_quantile_models = rpm_bundle["quantile_models"]
rpm_feature_names = rpm_bundle["feature_names"]

### Get Min predictions and Stat Per Min predictions 

In [8]:
pts_preds = predict_min_times_rate(
    pts_names, min_df, pts_df, current_date,
    rate_pipeline=ppm_pipeline,
    rate_quantile_models=ppm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="PTS",
)
ast_preds = predict_min_times_rate(
    ast_names, min_df, ast_df, current_date,
    rate_pipeline=apm_pipeline,
    rate_quantile_models=apm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="AST",
)
reb_preds = predict_min_times_rate(
    reb_names, min_df, reb_df, current_date,
    rate_pipeline=rpm_pipeline,
    rate_quantile_models=rpm_quantile_models,
    min_quantile_models=min_quantile_models,
    stat_prefix="REB",
)
pts_preds.head(10)

[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)
[SKIP] Wendell Carter Jr: min_pipeline returned None (need >= 10 games)
[SKIP] R.J. Barrett: min_pipeline returned None (need >= 10 games)


,PLAYER_NAME,MARKET,MIN_Q10,MIN_Q50,MIN_Q90,RATE_Q10,RATE_Q50,RATE_Q90,STAT_Q10,STAT_Q50,STAT_Q90,RATE_HISTORY
0,Cade Cunningham,PTS,29.21,37.78,42.31,0.3913,0.6315,0.8870,11.43,23.86,37.53,"[0.7754010695187166, 0.2580645161290322, 0.660..."
1,Paolo Banchero,PTS,29.29,38.11,41.49,0.3956,0.5835,0.8479,11.59,22.24,35.18,"[1.0617059891107077, 0.9007506255212676, 0.796..."
2,Desmond Bane,PTS,25.05,34.17,39.90,0.2799,0.4845,0.7472,7.01,16.56,29.81,"[0.5233453052847614, 0.467032967032967, 0.6616..."
3,Tobias Harris,PTS,23.46,33.52,39.83,0.2300,0.4548,0.6872,5.40,15.25,27.37,"[0.3186646433990895, 0.5172413793103449, 0.438..."
4,Franz Wagner,PTS,22.54,31.28,37.21,0.3934,0.6129,0.8687,8.87,19.17,32.32,"[0.9393346379647748, 0.9610983981693364, 0.674..."
5,Jalen Duren,PTS,19.68,26.37,33.24,0.2290,0.4488,0.6956,4.51,11.83,23.12,"[0.968392737054472, 1.07326178254783, 0.6, 0.6..."
6,Jalen Suggs,PTS,23.98,32.96,39.67,0.2250,0.5059,0.7912,5.39,16.68,31.39,"[0.4109589041095891, 0.4081632653061224, 0.235..."
7,Ausar Thompson,PTS,21.36,29.86,38.30,0.1310,0.3360,0.5833,2.80,10.03,22.34,"[0.1674730182359508, 0.4855460144764645, 0.379..."
8,Duncan Robinson,PTS,20.03,28.59,35.62,0.1420,0.3639,0.6117,2.84,10.40,21.79,"[0.3041825095057034, 0.5191594561186651, 0.378..."
9,Anthony Black,PTS,17.33,23.00,29.45,0.1795,0.4151,0.6589,3.11,9.55,19.40,"[0.5257836198179979, 0.6417112299465241, 0.222..."


### Get Line Probabilities

In [9]:
all_line_probs = pd.concat([
    line_probs_for_market(ast_preds, lines_dfs_ast, run_pts_simulation),
    line_probs_for_market(reb_preds, lines_dfs_reb, run_pts_simulation),
    line_probs_for_market(pts_preds, lines_dfs_pts, run_pts_simulation),
], ignore_index=True)
all_line_probs.sample(10)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER
20,Jalen Johnson,AST,5.5,25.15,36.00,42.68,1.62,4.76,8.84,0.453,0.547
28,VJ Edgecombe,AST,3.5,31.27,39.54,43.93,1.61,4.50,7.74,0.666,0.334
37,Jabari Smith Jr,AST,1.5,23.90,29.91,39.02,-0.00,0.66,3.32,0.288,0.712
117,Jakob Poeltl,PTS,6.5,14.60,20.61,29.41,2.46,7.38,17.98,0.647,0.352
142,Mikal Bridges,PTS,8.5,21.33,30.52,37.44,2.57,9.10,20.70,0.647,0.353
98,Tobias Harris,PTS,16.5,23.46,33.52,39.83,5.40,15.25,27.37,0.516,0.484
80,Derrick White,REB,4.0,22.55,28.60,35.93,0.78,3.00,6.72,0.303,0.697
64,Mitchell Robinson,REB,6.0,15.97,20.17,26.89,3.18,7.05,14.12,0.808,0.192
14,Rudy Gobert,AST,1.5,25.74,34.44,39.59,0.00,1.94,4.09,0.593,0.407
160,Bruce Brown,PTS,6.5,17.30,22.83,29.41,2.81,8.33,17.57,0.875,0.125


In [10]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Underdog',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

underdog_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
underdog_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
112,Collin Murray-Boyles,PTS,12.5,17.23,22.36,30.34,3.37,9.88,20.97,0.590,0.410,PTS,Underdog,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,-105.0,-114.0,0.512,0.533,14.4,16.0,6.28,1.9,3.5,-0.303,0.619,0.381,20.85,-28.48,0.8,0.7,0.53,0.25,22.41,4.60,0.20,0.08,13.00,6.0
107,Donovan Mitchell,PTS,26.5,28.20,36.49,41.41,13.06,26.04,40.59,0.568,0.432,PTS,Underdog,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-113.0,-104.0,0.531,0.510,24.1,27.5,10.87,-2.4,1.0,0.221,0.413,0.587,-22.15,15.14,0.6,0.5,0.53,0.61,33.03,3.83,0.29,0.07,24.17,6.0
170,Joel Embiid,PTS,27.5,23.18,28.75,35.24,13.23,23.24,36.77,0.396,0.604,PTS,Underdog,Boston Celtics,5.5,212.5,111.7,4.0,95.58,30.0,-125.0,-105.0,0.556,0.512,28.8,28.0,4.98,1.3,0.5,-0.261,0.603,0.397,8.54,-22.49,0.4,0.5,0.60,0.48,33.50,4.16,0.34,0.04,20.75,4.0
5,Donovan Mitchell,AST,4.5,28.20,36.49,41.41,1.69,4.30,7.61,0.520,0.480,AST,Underdog,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-110.0,-110.0,0.524,0.524,4.5,4.5,1.27,0.0,0.0,0.000,0.500,0.500,-4.55,-4.55,0.2,0.5,0.40,0.69,33.03,3.83,0.29,0.07,4.83,6.0
121,LeBron James,PTS,23.5,31.01,38.54,44.08,12.12,24.63,38.15,0.602,0.398,PTS,Underdog,Houston Rockets,-4.5,208.5,112.1,6.0,96.98,29.0,-112.0,-106.0,0.528,0.515,21.5,22.5,7.55,-2.0,-1.0,0.265,0.396,0.604,-25.04,17.38,0.4,0.5,0.33,0.36,33.17,7.98,0.30,0.08,21.71,7.0


In [11]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='PrizePicks',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

prizePicks_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
prizePicks_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
122,Alperen Sengun,PTS,22.5,22.85,28.62,38.02,8.20,16.13,31.57,0.309,0.691,PTS,PrizePicks,Los Angeles Lakers,4.5,208.5,115.5,20.0,99.22,22.0,-103.0,-114.0,0.507,0.533,20.1,19.5,6.87,-2.4,-3.0,0.349,0.364,0.636,-28.26,19.39,0.2,0.3,0.40,0.36,33.46,5.79,0.25,0.03,22.00,6.0
50,Dean Wade,REB,4.0,14.81,21.26,28.55,0.96,3.75,8.57,0.521,0.479,REB,PrizePicks,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-137.0,-137.0,0.578,0.578,4.0,4.5,1.76,0.0,0.5,0.000,0.500,0.500,-13.50,-13.50,0.6,0.5,0.53,0.44,22.77,3.66,0.08,0.04,4.17,6.0
14,Rudy Gobert,AST,1.5,25.74,34.44,39.59,0.00,1.94,4.09,0.593,0.407,AST,PrizePicks,Denver Nuggets,5.5,225.0,116.0,21.0,99.49,20.0,-103.0,-119.0,0.507,0.543,1.8,2.0,0.79,0.3,0.5,-0.380,0.648,0.352,27.71,-35.22,0.8,0.6,0.40,0.49,31.47,3.93,0.12,0.04,1.67,9.0
41,Desmond Bane,REB,4.0,25.05,34.17,39.90,1.25,4.30,8.55,0.638,0.362,REB,PrizePicks,Detroit Pistons,10.0,211.5,108.9,2.0,99.88,19.0,-137.0,-137.0,0.578,0.578,4.5,4.5,2.07,0.5,0.5,-0.242,0.596,0.404,3.10,-30.11,0.6,0.5,0.47,0.42,30.77,7.41,0.21,0.05,5.25,8.0
130,Tari Eason,PTS,12.5,22.93,33.04,41.50,3.98,13.17,27.02,0.555,0.445,PTS,PrizePicks,Los Angeles Lakers,4.5,208.5,115.5,20.0,99.22,22.0,-132.0,105.0,0.569,0.488,11.6,13.5,7.18,-0.9,1.0,0.125,0.450,0.550,-20.91,12.75,0.6,0.5,0.53,0.38,27.39,7.38,0.17,0.07,11.29,7.0


In [12]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='Betr DFS',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

betr_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
betr_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
171,Tyrese Maxey,PTS,24.5,29.61,38.43,45.13,13.05,25.75,42.47,0.650,0.350,PTS,Betr DFS,Boston Celtics,5.5,212.5,111.7,4.0,95.58,30.0,-122.0,105.0,0.550,0.488,24.3,23.5,5.19,-0.2,-1.0,0.039,0.484,0.516,-11.93,5.78,0.6,0.4,0.47,0.68,37.30,4.89,0.28,0.05,27.56,9.0
55,Amen Thompson,REB,7.5,22.89,30.60,41.15,2.25,5.68,12.51,0.329,0.671,REB,Betr DFS,Los Angeles Lakers,4.5,208.5,115.5,20.0,99.22,22.0,103.0,-118.0,0.493,0.541,7.1,7.5,2.81,-0.4,0.0,0.142,0.444,0.556,-9.87,2.72,0.4,0.5,0.53,0.52,40.46,4.39,0.20,0.04,8.14,7.0
160,Bruce Brown,PTS,6.5,17.30,22.83,29.41,2.81,8.33,17.57,0.875,0.125,PTS,Betr DFS,Minnesota Timberwolves,-5.5,225.0,112.5,8.0,101.50,10.0,-110.0,-105.0,0.524,0.512,8.5,7.5,3.03,2.0,1.0,-0.660,0.745,0.255,42.23,-50.21,0.8,0.7,0.73,0.59,21.69,5.05,0.18,0.05,6.56,9.0
9,Jalen Brunson,AST,6.5,29.04,38.00,42.23,2.45,5.69,10.11,0.476,0.524,AST,Betr DFS,Atlanta Hawks,-2.0,213.5,112.9,10.0,102.50,5.0,-108.0,100.0,0.519,0.500,7.2,7.5,3.43,0.7,1.0,-0.204,0.581,0.419,11.90,-16.20,0.6,0.7,0.67,0.54,35.48,3.25,0.31,0.05,6.50,8.0
11,Ayo Dosunmu,AST,4.5,22.99,32.38,39.59,1.16,4.31,9.62,0.548,0.452,AST,Betr DFS,Denver Nuggets,5.5,225.0,116.0,21.0,99.49,20.0,-105.0,-110.0,0.512,0.524,3.4,3.5,2.46,-1.1,-1.0,0.447,0.327,0.673,-36.16,28.48,0.4,0.2,0.40,0.31,32.48,5.83,0.21,0.05,4.14,7.0


In [13]:
pra_lines_dfs = pd.concat([lines_dfs_pts, lines_dfs_ast, lines_dfs_reb])
pra_lines_us = pd.concat([lines_us_pts, lines_us_ast, lines_us_reb])

final, tier1_all, final = generalized_best_bets(
    pra_lines_dfs, base_df, pra_lines_us, team_dds, nameDict,
    line_bookmaker='DraftKings Pick6',
)
df = final
prop_label_map = {
    'player_points': 'PTS',
    'player_rebounds': 'REB',
    'player_assists': 'AST',
    'player_turnovers': 'TOV',
    'player_frees_attempts': 'FTA',
    'player_threes': '3PM',
    'player_blocks': 'BLK',
    'player_steals': 'STL',
    'player_blocks_steals': 'BLK+STL',
    'player_points_rebounds_assists': 'PTS+REB+AST',
    'player_points_rebounds': 'PTS+REB',
    'player_points_assists': 'PTS+AST',
    'player_rebounds_assists': 'REB+AST',
}

# In this cell `df['Prop']` is the CATEGORY value (e.g., 'player_points').
df['CATEGORY'] = df['CATEGORY'].map(prop_label_map).fillna(df['CATEGORY'])
df.drop(columns=['LINE'], inplace=True)

draftKings_all_lines = all_line_probs.merge(df, left_on=['PLAYER_NAME', 'MARKET'], right_on=['PLAYER_NAME', 'CATEGORY'], how='left').dropna()
draftKings_all_lines.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
7,Amen Thompson,AST,5.5,22.89,30.60,41.15,1.34,3.79,8.24,0.380,0.620,AST,DraftKings Pick6,Los Angeles Lakers,4.5,208.5,115.5,20.0,99.22,22.0,-110.0,108.0,0.524,0.481,6.3,7.0,1.77,0.8,1.5,-0.452,0.674,0.326,28.67,-32.19,0.8,0.7,0.60,0.48,40.46,4.39,0.20,0.04,5.86,7.0
35,Brandon Ingram,AST,3.5,27.75,36.46,41.20,0.97,3.11,6.48,0.471,0.529,AST,DraftKings Pick6,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,116.0,-135.0,0.463,0.574,3.6,3.5,2.07,0.1,0.0,-0.048,0.519,0.481,12.10,-16.27,0.2,0.5,0.40,0.53,32.96,3.68,0.24,0.05,2.86,7.0
109,Scottie Barnes,PTS,19.5,28.08,36.96,41.31,7.19,17.65,30.14,0.587,0.413,PTS,DraftKings Pick6,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,-104.0,-107.0,0.510,0.517,19.4,19.5,7.56,-0.1,0.0,0.013,0.495,0.505,-2.90,-2.30,0.8,0.5,0.47,0.38,32.52,5.74,0.24,0.06,23.29,7.0
4,James Harden,AST,6.5,29.29,38.35,42.79,2.95,7.21,11.86,0.499,0.501,AST,DraftKings Pick6,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,-118.0,105.0,0.541,0.488,6.4,5.0,3.44,-0.1,-1.5,0.029,0.488,0.512,-9.84,4.96,0.4,0.4,0.60,0.70,33.46,4.20,0.27,0.06,7.20,5.0
115,Ja'Kobe Walter,PTS,7.5,18.24,24.03,31.30,1.78,7.20,17.55,0.585,0.415,PTS,DraftKings Pick6,Cleveland Cavaliers,9.0,216.0,114.1,15.0,100.70,13.0,-104.0,-105.0,0.510,0.512,8.6,8.0,5.64,1.1,0.5,-0.195,0.577,0.423,13.18,-17.41,0.4,0.5,0.53,0.41,27.37,2.27,0.13,0.03,4.71,7.0


In [14]:
all_line_probs = pd.concat([underdog_all_lines, prizePicks_all_lines, betr_all_lines, draftKings_all_lines])
all_line_probs.to_json('data/props/ev_analysis/all_line_probs.json', orient='records', lines=True)
all_line_probs.sample(5)

,PLAYER_NAME,MARKET,LINE,MIN_Q10,MIN_Q50,MIN_Q90,STAT_Q10,STAT_Q50,STAT_Q90,P_OVER,P_UNDER,CATEGORY,LINE_BOOKMAKER,OPPONENT,TEAM_SPREAD,GAME_TOTAL,OPP_DEF_RATING,OPP_RANK_DEF_RATING,OPP_PACE,OPP_PACE_RANK,ODDS_OVER,ODDS_UNDER,IMP_PROB_OVER,IMP_PROB_UNDER,AVG_STAT_L10,MED_STAT_L10,STD_STAT_L10,EDGE,MED_EDGE,Z_SCORE,PROB_OVER,PROB_UNDER,EV_OVER,EV_UNDER,OVER_RATE_L5,OVER_RATE_L10,OVER_RATE_L15,OVER_RATE_SEASON,AVG_MIN_L10,STD_MIN_L10,AVG_USG_L10,STD_USG_L10,AVG_STAT_VS_MATCHUP,MATCHUP_GAMES
141,Josh Hart,PTS,11.5,27.62,36.72,41.61,3.09,11.00,21.50,0.488,0.512,PTS,Betr DFS,Atlanta Hawks,-2.0,213.5,112.9,10.0,102.50,5.0,100.0,-114.0,0.500,0.533,9.0,8.0,7.33,-2.5,-3.5,0.341,0.367,0.633,-26.60,18.83,0.2,0.2,0.40,0.49,32.18,4.49,0.14,0.06,8.17,6.0
127,Deandre Ayton,PTS,12.5,21.87,30.52,37.53,4.10,11.69,24.10,0.493,0.506,PTS,Underdog,Houston Rockets,-4.5,208.5,112.1,6.0,96.98,29.0,-104.0,-110.0,0.510,0.524,12.3,11.5,7.57,-0.2,-1.0,0.026,0.490,0.510,-3.88,-2.64,0.6,0.5,0.47,0.46,26.82,5.31,0.18,0.06,11.29,7.0
113,Jarrett Allen,PTS,11.5,20.61,29.00,35.36,4.71,13.50,25.45,0.744,0.256,PTS,DraftKings Pick6,Toronto Raptors,-9.0,216.0,112.1,5.0,99.22,21.0,100.0,100.0,0.500,0.500,12.4,12.5,4.81,0.9,1.0,-0.187,0.574,0.426,14.80,-14.80,0.4,0.6,0.67,0.63,24.22,3.89,0.19,0.07,8.20,5.0
73,LeBron James,REB,7.5,31.01,38.54,44.08,3.43,7.81,13.96,0.721,0.279,REB,Betr DFS,Houston Rockets,-4.5,208.5,112.1,6.0,96.98,29.0,102.0,-110.0,0.495,0.524,7.1,7.0,2.73,-0.4,-0.5,0.147,0.442,0.558,-10.72,6.53,0.6,0.5,0.53,0.30,33.17,7.98,0.30,0.08,6.43,7.0
136,Jalen Johnson,PTS,19.5,25.15,36.00,42.68,6.34,17.45,32.03,0.482,0.518,PTS,Underdog,New York Knicks,2.0,213.5,112.3,7.0,97.71,25.0,-117.0,-106.0,0.539,0.515,18.3,18.0,3.68,-1.2,-1.5,0.326,0.372,0.628,-31.01,22.05,0.4,0.3,0.47,0.62,34.31,4.85,0.25,0.03,19.38,8.0


### Get top EVs for 2 legs

In [15]:
slate_path = build_greedy_slate(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks.json",
)
print(slate_path)

Legs: 81  |  Pairs: 98  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks.json


In [16]:
slate_path = build_greedy_slate(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog.json",
)
print(slate_path)

Legs: 49  |  Pairs: 52  |  Slate: 2  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog.json


In [17]:
slate_path = build_greedy_slate(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings.json",
)
print(slate_path)

Legs: 49  |  Pairs: 51  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings.json


In [18]:
slate_path = build_greedy_slate(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr.json",
)
print(slate_path)

Legs: 118  |  Pairs: 194  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr.json


### Top EVs for 3 Legs

In [19]:
slate_path = build_greedy_slate_3leg(
    prob_df=prizePicks_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/prizepicks_3leg.json",
)
print(slate_path)

Legs: 81  |  Triples: 1166  |  Slate: 4  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/prizepicks_3leg.json


In [20]:
slate_path = build_greedy_slate_3leg(
    prob_df=underdog_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/underdog_3leg.json",
)
print(slate_path)

Legs: 49  |  Triples: 428  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/underdog_3leg.json


In [21]:
slate_path = build_greedy_slate_3leg(
    prob_df=betr_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/betr_3leg.json",
)
print(slate_path)

Legs: 118  |  Triples: 3696  |  Slate: 6  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/betr_3leg.json


In [22]:
slate_path = build_greedy_slate_3leg(
    prob_df=draftKings_all_lines,
    min_df=min_df,
    min_ev=0.5,          # only pairs with EV > 50%
    min_kelly=0.10,      # only pairs with Kelly > 10%
    kelly_fraction=0.5,  # half Kelly
    top_n=10,
    json_path="data/props/ev_analysis/draftKings_3leg.json",
)
print(slate_path)

Legs: 49  |  Triples: 347  |  Slate: 3  |  JSON: /Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
/Users/alexgonzalez/Documents/NBA-Prop-Predictor/data/props/ev_analysis/draftKings_3leg.json
